In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

100%|██████████| 26.4M/26.4M [00:05<00:00, 4.87MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 75.2kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.53MB/s]
100%|██████████| 5.15k/5.15k [00:00<?, ?B/s]


In [2]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [6]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

In [10]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

epochs = 5
for t in range(epochs):
    train(train_dataloader, model, loss_fn, optimizer)

loss: 2.298394  [   64/60000]
loss: 2.286469  [ 6464/60000]
loss: 2.265791  [12864/60000]
loss: 2.261885  [19264/60000]
loss: 2.240937  [25664/60000]
loss: 2.222320  [32064/60000]
loss: 2.226117  [38464/60000]
loss: 2.194339  [44864/60000]
loss: 2.189139  [51264/60000]
loss: 2.155609  [57664/60000]
loss: 2.160911  [   64/60000]
loss: 2.144706  [ 6464/60000]
loss: 2.088981  [12864/60000]
loss: 2.105496  [19264/60000]
loss: 2.034468  [25664/60000]
loss: 1.994765  [32064/60000]
loss: 2.012137  [38464/60000]
loss: 1.934654  [44864/60000]
loss: 1.950219  [51264/60000]
loss: 1.856681  [57664/60000]
loss: 1.900259  [   64/60000]
loss: 1.862833  [ 6464/60000]
loss: 1.744445  [12864/60000]
loss: 1.783143  [19264/60000]
loss: 1.655861  [25664/60000]
loss: 1.624160  [32064/60000]
loss: 1.636001  [38464/60000]
loss: 1.541349  [44864/60000]
loss: 1.576589  [51264/60000]
loss: 1.455156  [57664/60000]
loss: 1.554137  [   64/60000]
loss: 1.517807  [ 6464/60000]
loss: 1.366709  [12864/60000]
loss: 1.43

In [3]:
import numpy as np
data = np.load(r'I:\HSP\EHR\I0002\ehr_labels_v1.npz')
